# Bloque 4: Validación Robusta
## Cómo evaluar modelos en producción sin sorpresas

**Objetivo**: Entender por qué la validación importa, cuál estrategia usar, cómo detectar problemas antes de deployar.

**Tiempo estimado**: 45 minutos de lectura + ejecución.

**Sin obviedades**: Asume conocimiento de ML base. Orientado a decisiones reales en producción.

---
## Setup: Imports y Configuración para Colab

In [ ]:
# Imports esenciales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    KFold, 
    StratifiedKFold,
    TimeSeriesSplit
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, 
    roc_curve, 
    accuracy_score,
    precision_recall_curve,
    f1_score,
    confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.style.use('default')
np.random.seed(42)

print("✓ Todos los imports listos para Colab")

---
## Dataset: Aprobación de Créditos en Banca

Caso real: Banco automático decide si aprobar crédito a un cliente.

- **2500 clientes**, 30 features (ingresos, edad, historial, monto, etc.)
- **Desbalance**: 75% aprobado, 25% rechazado
- **Reto**: Modelo debe ser justo y confiable en diferentes grupos

In [ ]:
# Crear dataset realista
X, y = make_classification(
    n_samples=2500,
    n_features=30,
    n_informative=18,
    n_redundant=7,
    n_classes=2,
    weights=[0.75, 0.25],  # Desbalance: 75% aprobado, 25% rechazado
    random_state=42,
    flip_y=0.05
)

# Split inicial
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalar
scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full)
X_test = scaler.transform(X_test)

print(f"Dataset total: {X.shape[0]} clientes")
print(f"\nTrain+Val: {X_train_full.shape[0]} (usaremos para validación cruzada)")
print(f"Test: {X_test.shape[0]} (evaluación FINAL, nunca toca entrenamiento)")
print(f"\nDesbalance: {(y==1).sum()} rechazados ({y.mean():.1%}) de {len(y)} total")

---
# SECCIÓN 1: ¿Por qué validación importa?

**La pregunta**: ¿Puedo confiar en una métrica de un solo split de datos?

**Respuesta**: NO. Un solo split puede ser suerte (o mala suerte). La validación cruzada responde: "¿Cómo performa el modelo en DIFERENTES subconjuntos de datos?"

In [ ]:
# Escenario: Evaluamos el MISMO modelo en 5 splits DIFERENTES
print("DEMOSTRACION: Mismo modelo, 5 splits diferentes\n")

model = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)

# Entrenamos en train_full y evaluamos en splits diferentes
aucs_different_splits = []

for split_idx in range(5):
    # Crear split diferente manualmente
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train_full, y_train_full, test_size=0.25, random_state=42+split_idx, stratify=y_train_full
    )
    
    # Entrenar
    model.fit(X_train_split, y_train_split)
    
    # Evaluar en val_split
    auc = roc_auc_score(y_val_split, model.predict_proba(X_val_split)[:, 1])
    aucs_different_splits.append(auc)
    print(f"Split {split_idx+1}: AUC = {auc:.4f}")

print(f"\n" + "="*70)
print(f"AUC mínimo: {min(aucs_different_splits):.4f}")
print(f"AUC máximo: {max(aucs_different_splits):.4f}")
print(f"AUC promedio: {np.mean(aucs_different_splits):.4f}")
print(f"AUC desviación estándar: {np.std(aucs_different_splits):.4f}")
print(f"\n⚠️  Diferencia de {(max(aucs_different_splits) - min(aucs_different_splits)):.4f} entre splits!")
print(f"\n→ Si solo miras UN split, podrías pensar que AUC = {max(aucs_different_splits):.4f}")
print(f"→ Pero en otro split podría ser = {min(aucs_different_splits):.4f}")
print(f"→ Cross-validation te da la VERDADERA performance: {np.mean(aucs_different_splits):.4f} ± {np.std(aucs_different_splits):.4f}")
print("="*70)

### ✅ CONCLUSIÓN Sección 1

**Un solo split te engaña**: Puede haber suerte (AUC alto) o mala suerte (AUC bajo).

**Cross-Validation te da la verdad**: Promedias múltiples splits → estimación confiable de cómo performa en datos NO vistos.

**Regla de oro**: NUNCA reportes métrica de un solo split. Siempre usa K-Fold (típicamente K=5).

---
# SECCIÓN 2: Cross-Validation - 4 Variantes

Hay diferentes formas de dividir datos. Cada una es mejor en contextos específicos.

## 2.1: Hold-Out (Simple pero arriesgado)

**Idea**: Divide una sola vez en Train/Val.

**Ventaja**: Rápido, simple.

**Desventaja**: Una sola división → métrica puede no ser representativa.

**Cuándo usar**: Datasets ENORMES (>100K muestras) donde CV tarda demasiado.

In [ ]:
# Hold-Out: dividir UNA sola vez
X_train_ho, X_val_ho, y_train_ho, y_val_ho = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)

model_ho = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)
model_ho.fit(X_train_ho, y_train_ho)

auc_holdout = roc_auc_score(y_val_ho, model_ho.predict_proba(X_val_ho)[:, 1])

print("\n2.1: HOLD-OUT VALIDATION")
print(f"Train size: {X_train_ho.shape[0]}")
print(f"Val size: {X_val_ho.shape[0]}")
print(f"AUC: {auc_holdout:.4f}")
print(f"\n⚠️  Métrica de UNA división (puede tener suerte o mala suerte)")

## 2.2: K-Fold Cross-Validation (Robusto)

**Idea**: Divide datos en K folds. Cada fold es val UNA VEZ, train (K-1) veces. Promedia métricas.

**Ventaja**: Robusto, cada dato participa en train y val.

**Desventaja**: Puede mezclar clases (si datos están ordenados).

**Cuándo usar**: Datos desordenados, pequeño-mediano dataset.

In [ ]:
# K-Fold: 5 splits
kf = KFold(n_splits=5, shuffle=True, random_state=42)

aucs_kfold = []
for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_full)):
    X_train_fold = X_train_full[train_idx]
    X_val_fold = X_train_full[val_idx]
    y_train_fold = y_train_full[train_idx]
    y_val_fold = y_train_full[val_idx]
    
    model = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)
    model.fit(X_train_fold, y_train_fold)
    
    auc = roc_auc_score(y_val_fold, model.predict_proba(X_val_fold)[:, 1])
    aucs_kfold.append(auc)
    print(f"Fold {fold_idx+1}: AUC = {auc:.4f}")

auc_kfold_mean = np.mean(aucs_kfold)
auc_kfold_std = np.std(aucs_kfold)

print(f"\n2.2: K-FOLD CROSS-VALIDATION (K=5)")
print(f"AUC promedio: {auc_kfold_mean:.4f} ± {auc_kfold_std:.4f}")
print(f"\n✓ Robusto: promedia 5 divisiones diferentes")
print(f"✓ Intervalo de confianza: AUC está entre {auc_kfold_mean - auc_kfold_std:.4f} y {auc_kfold_mean + auc_kfold_std:.4f}")

## 2.3: Stratified K-Fold (Recomendado para desbalance)

**Idea**: K-Fold pero garantiza que cada fold tiene MISMA proporción de clases.

**Ventaja**: Perfecto para datos desbalanceados (10% fraude, 90% legítimo).

**Cuándo usar**: SIEMPRE que tengas desbalance (99% de casos reales).

**Por qué**: Evita que un fold tenga 0% de la clase minoritaria (métrica sin sentido).

In [ ]:
# Stratified K-Fold: mantiene proporción de clases
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

aucs_skfold = []
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):
    X_train_fold = X_train_full[train_idx]
    X_val_fold = X_train_full[val_idx]
    y_train_fold = y_train_full[train_idx]
    y_val_fold = y_train_full[val_idx]
    
    model = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)
    model.fit(X_train_fold, y_train_fold)
    
    auc = roc_auc_score(y_val_fold, model.predict_proba(X_val_fold)[:, 1])
    aucs_skfold.append(auc)
    
    # Mostrar proporción de clases en este fold
    prop_rechazados = y_val_fold.mean()
    print(f"Fold {fold_idx+1}: AUC = {auc:.4f} | Proporción rechazo en val = {prop_rechazados:.1%}")

auc_skfold_mean = np.mean(aucs_skfold)
auc_skfold_std = np.std(aucs_skfold)

print(f"\n2.3: STRATIFIED K-FOLD CROSS-VALIDATION (K=5)")
print(f"AUC promedio: {auc_skfold_mean:.4f} ± {auc_skfold_std:.4f}")
print(f"\n✓ Cada fold tiene {y_train_full.mean():.1%} de rechazo (balanceado)")
print(f"✓ Métricas confiables incluso con desbalance")

## 2.4: Time Series Split (Para datos temporales)

**Idea**: Divide cronológicamente. Train = pasado, Val = futuro (nunca entrena en futuro).

**Ventaja**: Previene leakage temporal (info del futuro filtrando al pasado).

**Cuándo usar**: SIEMPRE que tengas serie temporal (stock prices, clickthrough, defaults por mes).

**Por qué**: Simula realidad: hoy entrenas con datos pasados, predices futuro.

In [ ]:
# Time Series Split: simula que train = pasado, val = futuro
tscv = TimeSeriesSplit(n_splits=5)

aucs_tscv = []
for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X_train_full)):
    X_train_fold = X_train_full[train_idx]
    X_val_fold = X_train_full[val_idx]
    y_train_fold = y_train_full[train_idx]
    y_val_fold = y_train_full[val_idx]
    
    model = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)
    model.fit(X_train_fold, y_train_fold)
    
    auc = roc_auc_score(y_val_fold, model.predict_proba(X_val_fold)[:, 1])
    aucs_tscv.append(auc)
    
    print(f"Split {fold_idx+1}: Train=índices 0-{train_idx.max()}, Val=índices {val_idx.min()}-{val_idx.max()} | AUC = {auc:.4f}")

auc_tscv_mean = np.mean(aucs_tscv)
auc_tscv_std = np.std(aucs_tscv)

print(f"\n2.4: TIME SERIES SPLIT (K=5)")
print(f"AUC promedio: {auc_tscv_mean:.4f} ± {auc_tscv_std:.4f}")
print(f"\n✓ Train siempre ANTES de Val (temporal)")
print(f"✓ Simula predicción en tiempo real")

### Comparación: Las 4 variantes lado a lado

In [ ]:
# Tabla comparativa
comparison_cv = pd.DataFrame({
    'Método': ['Hold-Out', 'K-Fold', 'Stratified K-Fold', 'Time Series Split'],
    'AUC': [f"{auc_holdout:.4f}", f"{auc_kfold_mean:.4f}", f"{auc_skfold_mean:.4f}", f"{auc_tscv_mean:.4f}"],
    'Desv. Est.': ["—", f"± {auc_kfold_std:.4f}", f"± {auc_skfold_std:.4f}", f"± {auc_tscv_std:.4f}"],
    'Casos de uso': ['Datasets enormes', 'Datos desordenados', 'Datos desbalanceados (RECOMENDADO)', 'Series temporales']
})

print("\n" + "="*100)
print("COMPARACIÓN: Cuál Cross-Validation usar")
print("="*100)
print(comparison_cv.to_string(index=False))
print("="*100)

### Visualización: Distribución de AUC por método

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

methods = ['Hold-Out\n(1 split)', 'K-Fold\n(5 splits)', 'Stratified K-Fold\n(5 splits)', 'Time Series\n(5 splits)']
all_aucs = [[auc_holdout], aucs_kfold, aucs_skfold, aucs_tscv]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# Box plot
box_data = []
labels = []
for i, (method, aucs, color) in enumerate(zip(methods, all_aucs, colors)):
    positions = [i]
    bp = ax.boxplot(aucs, positions=positions, widths=0.4, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=2),
                    whiskerprops=dict(color='black'),
                    capprops=dict(color='black'))
    labels.append(method)

ax.set_xticks(range(len(methods)))
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('AUC', fontsize=12, fontweight='bold')
ax.set_title('Variabilidad de AUC por método de Cross-Validation', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0.75, 0.85])

plt.tight_layout()
plt.savefig('cv_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico guardado como 'cv_comparison.png'")

### ✅ CONCLUSIÓN Sección 2

| Caso | Recomienda | Razón |
|------|-----------|-------|
| Datos grandes (>100K) | Hold-Out | Speed |
| Datos pequeños, balanceados | K-Fold | Robusted |
| Datos desbalanceados | **Stratified K-Fold** | Proporciones iguales |
| Series temporales | Time Series Split | No leakage temporal |

**Recomendación**: En 90% de casos reales usa **Stratified K-Fold (K=5)**.

---
# SECCIÓN 4: Métricas Apropiadas + Out-Of-Distribution (OOD)

**La pregunta**: ¿AUC es siempre la mejor métrica? ¿Qué pasa cuando llegan datos de otra distribución?

## 4.1: Cuándo usar cada métrica

**AUC-ROC**: Mide discriminación general. Insensible a umbral.
- Úsalo cuando: Importa ranking (cliente de mayor riesgo primero)

**Accuracy**: % predicciones correctas.
- ⚠️  PELIGRO con desbalance (80% accuracy puede ser un modelo tonto que dice "aprobado" a todos)

**Precision-Recall**: Precisión = "de los que dije rechazados, cuántos eran reales?". Recall = "de los rechazos reales, cuántos encontré?"
- Úsalo cuando: Costo de falso positivo ≠ costo de falso negativo

**F1-Score**: Promedio armónico de Precision y Recall.
- Úsalo cuando: Quieres balance entre ambas

In [ ]:
# Entrenar modelo en train_full
model_final = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
model_final.fit(X_train_full, y_train_full)

# Predicciones en test
y_pred_proba = model_final.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

# Calcular métricas
auc = roc_auc_score(y_test, y_pred_proba)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print("4.1: METRICAS EN TEST SET (Modelo en distribución original)\n")
print(f"AUC-ROC: {auc:.4f}")
print(f"  → Mide discriminación. Independiente del umbral.")
print(f"\nAccuracy: {accuracy:.4f}")
print(f"  → % predicciones correctas. ⚠️  Engañoso con desbalance.")
print(f"\nPrecision: {precision:.4f}")
print(f"  → De {tp + fp} que dije 'rechazados', {tp} eran reales.")
print(f"\nRecall: {recall:.4f}")
print(f"  → De {tp + fn} rechazos reales, encontré {tp}.")
print(f"\nF1-Score: {f1:.4f}")
print(f"  → Promedio armónico de Precision-Recall.")
print(f"\n" + "="*70)
print(f"Confusion Matrix:")
print(f"  TN={tn} (aprobados bien)  | FP={fp} (aprobados mal)")
print(f"  FN={fn} (rechazados mal)  | TP={tp} (rechazados bien)")
print("="*70)

## 4.2: Out-Of-Distribution (OOD) - Cambio en distribución

**El problema**: El banco entrena con clientes en Lima. Después expande a Cusco.
Cusco tiene distribución DIFERENTE de ingresos, edad, historial.

**Síntoma**: AUC cae bruscamente en nuevos datos → OOD.

**Solución**: Monitorear métricas en nuevos clientes, recalibrar modelo si cae > X%.

In [ ]:
# Simular distribución diferente (OOD)
# "Clientes de Cusco" tienen características distintas
np.random.seed(123)
X_ood = X_test.copy()
X_ood = X_ood + np.random.normal(0.8, 0.4, X_ood.shape)  # Shift en características

# Mantener mismos targets para evaluación
y_ood = y_test.copy()

# Predicciones en distribución OOD
y_pred_proba_ood = model_final.predict_proba(X_ood)[:, 1]
y_pred_ood = (y_pred_proba_ood > 0.5).astype(int)

# Métricas en OOD
auc_ood = roc_auc_score(y_ood, y_pred_proba_ood)
accuracy_ood = accuracy_score(y_ood, y_pred_ood)
f1_ood = f1_score(y_ood, y_pred_ood)

print("\n4.2: EVALUACION OUT-OF-DISTRIBUTION (OOD)\n")
print("Simulamos: Clientes de nueva región (Cusco) con distribución diferente\n")
print(f"Métrica             | En-Distribución | Out-Of-Dist | Caída")
print(f"-" * 65)
print(f"AUC-ROC             | {auc:.4f}          | {auc_ood:.4f}      | {(1 - auc_ood/auc)*100:+.1f}%")
print(f"Accuracy            | {accuracy:.4f}          | {accuracy_ood:.4f}      | {(1 - accuracy_ood/accuracy)*100:+.1f}%")
print(f"F1-Score            | {f1:.4f}          | {f1_ood:.4f}      | {(1 - f1_ood/f1)*100:+.1f}%")
print(f"\n" + "="*65)
print(f"⚠️  AUC cae {(1 - auc_ood/auc)*100:.1f}% en datos OOD → modelo frágil ante distribution shift")
print(f"\n✓ Solución: Monitorear AUC en datos nuevos, recalibrar si cae > 5%")

### Visualización: Performance en distribución vs OOD

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Comparación de métricas
metrics_names = ['AUC', 'Accuracy', 'F1']
in_dist = [auc, accuracy, f1]
ood_dist = [auc_ood, accuracy_ood, f1_ood]

x = np.arange(len(metrics_names))
width = 0.35

ax = axes[0]
bars1 = ax.bar(x - width/2, in_dist, width, label='In-Distribution (Lima)', alpha=0.8, color='#2ca02c')
bars2 = ax.bar(x + width/2, ood_dist, width, label='Out-Of-Distribution (Cusco)', alpha=0.8, color='#d62728')

ax.set_ylabel('Métrica', fontsize=11, fontweight='bold')
ax.set_title('Performance: In-Distribution vs OOD', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend(fontsize=10)
ax.set_ylim([0.7, 0.9])
ax.grid(True, alpha=0.3, axis='y')

# Agregar valores en barras
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom', fontsize=9)

# Panel 2: Distribución de scores
ax = axes[1]
ax.hist(y_pred_proba[y_test == 0], bins=20, alpha=0.6, label='Aprobados (In-Dist)', color='#2ca02c', edgecolor='black')
ax.hist(y_pred_proba[y_test == 1], bins=20, alpha=0.6, label='Rechazados (In-Dist)', color='#1f77b4', edgecolor='black')
ax.hist(y_pred_proba_ood[y_ood == 0], bins=20, alpha=0.3, label='Aprobados (OOD)', color='#2ca02c', edgecolor='red', linestyle='--')
ax.hist(y_pred_proba_ood[y_ood == 1], bins=20, alpha=0.3, label='Rechazados (OOD)', color='#1f77b4', edgecolor='red', linestyle='--')

ax.set_xlabel('Score de Probabilidad', fontsize=11, fontweight='bold')
ax.set_ylabel('Frecuencia', fontsize=11, fontweight='bold')
ax.set_title('Distribución de Scores: In-Dist vs OOD', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('metrics_ood.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico guardado como 'metrics_ood.png'")

### ✅ CONCLUSIÓN Sección 4

**Cuándo usar cada métrica**:
- Desbalance severo → **Precision-Recall o F1**
- Ranking importante → **AUC**
- Análisis rápido → **Accuracy** (pero cuidado con desbalance)

**Out-Of-Distribution (OOD)**:
- Siempre monitorea performance en datos NUEVOS
- Si AUC cae > 5% → señal de distribution shift
- Solución: Recalibración, reentrenamiento o alert

**Regla de oro**: No confíes en métricas de UN dataset. Prueba en múltiples distribuciones.

---
# SECCIÓN 5: Checklist + Flujo de Trabajo Recomendado

## Flujo de Trabajo: Validación Robusta en Producción

```
PASO 1: Preparar datos
  ├── Train/Val (80%): Para validación cruzada
  └── Test (20%): NUNCA participa en tuning ni validación

PASO 2: Elegir CV strategy
  ├── ¿Datos desbalanceados? → Stratified K-Fold ✓ RECOMENDADO
  ├── ¿Datos desordenados? → K-Fold
  ├── ¿Serie temporal? → Time Series Split
  └── ¿Dataset > 100K? → Hold-Out

PASO 3: Entrenar modelo
  └── Usar CV en train/val → reporta AUC ± desviación
     NO un AUC de un solo split

PASO 4: Elegir métrica
  ├── Desbalance → F1 o Precision-Recall
  ├── Ranking → AUC
  └── Mixto → AUC + F1

PASO 5: Evaluar en Test LIMPIO
  └── Métrica final en datos nunca vistos

PASO 6: Monitorear en producción
  ├── Medir métrica en rolling windows (cada semana)
  ├── ¿AUC cae > 5%? → Distribution shift
  └── Recalibrar o reentre ar si necesario
```

## Checklist: ¿Tu validación es robusta?

In [ ]:
# Checklist interactivo
checklist = {
    '✓ Usas K-Fold (≥5) o Stratified K-Fold': 
        'No reportas métrica de UN solo split.',
    
    '✓ Test set está COMPLETAMENTE APARTE': 
        'No participa en tuning ni validación.',
    
    '✓ Reportas métrica ± desviación estándar': 
        'AUC = 0.82 ± 0.03 (no solo 0.82).',
    
    '✓ Usas métrica apropiada': 
        'Desbalance → F1. Ranking → AUC.',
    
    '✓ Evaluaste en datos OOD': 
        'Cómo performa con nueva distribución.',
    
    '✓ Plan de monitoreo en producción': 
        'Verificarás métrica cada semana.'
}

print("\n" + "="*90)
print("CHECKLIST: ¿Tu validación es robusta?")
print("="*90)

for i, (check, description) in enumerate(checklist.items(), 1):
    print(f"\n{i}. {check}")
    print(f"   → {description}")

print("\n" + "="*90)
print("⚠️  Si NO checkeaste 5+ items → hay riesgo de sorpresas en producción.")
print("✅ Si checkeaste todos → confianza en performance real.")
print("="*90)

---
# RESUMEN EJECUTIVO

## Lo que aprendiste

| Sección | Concepto clave | Impacto |
|---------|----------------|--------|
| **¿Por qué validación?** | Un split engaña. CV te da verdad | Confianza en performance |
| **Hold-Out** | Rápido pero arriesgado | Solo para datasets enormes |
| **K-Fold** | Robusto, cada dato en train y val | Estándar en ML |
| **Stratified K-Fold** | Mantiene proporciones de clases | RECOMENDADO (99% casos) |
| **Time Series Split** | Train = pasado, Val = futuro | Esencial para temporales |
| **Métricas** | AUC vs Accuracy vs F1 | Elige según contexto |
| **OOD** | Performance con nueva distribución | Previene sorpresas |
| **Monitoreo** | Verificar métricas en producción | Detectar drift temprano |

## Reglas de Oro

1. **NUNCA reportes métrica de un solo split** → Usa K-Fold ≥5
2. **Datos desbalanceados** → Usa Stratified K-Fold
3. **Datos temporales** → Usa Time Series Split
4. **Test set es sagrado** → Nunca lo toques hasta la evaluación final
5. **Monitorea en producción** → AUC cada semana, alerta si cae > 5%
6. **Documenta decisiones** → Por qué usaste métrica X, CV strategy Y

## Herramientas recomendadas

```python
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='roc_auc')
print(f"AUC = {scores.mean():.4f} ± {scores.std():.4f}")
```

Eso es todo lo que necesitas para validación robusta en producción.